# CT-HTTPS v3 – full experiment run (Google Colab)

**Run this notebook on your own Colab account** (Runtime → Change runtime type → **GPU**).
Your MIMIC-III `CHARTEVENTS.csv` stays in **your** Google Drive; nothing is uploaded anywhere else.

1. Put `ct_https_v3.zip` and `CHARTEVENTS.csv` in your Google Drive (e.g. `MyDrive/ct_https/`).
2. Edit the two paths in the next cell.
3. Runtime → **Run all**. If Colab disconnects, just **Run all** again: finished runs are skipped.
4. At the end, share **only** `results_summary.zip` (aggregate numbers, no patient data).
   Never share the `results/preds`, `results/ckpt` or `cache` folders – they contain patient-level data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ct_https'          # <-- folder with ct_https_v3.zip
CSV       = '/content/drive/MyDrive/ct_https/CHARTEVENTS.csv'   # <-- full MIMIC-III CHARTEVENTS
SEEDS     = '0 1 2'                                       # 5 seeds (0 1 2 3 4) is better if time allows
MAXPP     = 20     # samples (label times) per patient; SAME value for every run. Lower it if runs are too slow

import os
WORK = f'{DRIVE_DIR}/work'                                # results are written to Drive (survive disconnects)
os.makedirs(WORK, exist_ok=True)
assert os.path.isfile(CSV), CSV

In [ ]:
!rm -rf /content/ct_https && mkdir -p /content/ct_https && unzip -q -o "$DRIVE_DIR/ct_https_v3.zip" -d /content/ct_https
%cd /content/ct_https
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Sanity tests (synthetic data, ~5 min)

In [ ]:
!python tests/test_pipeline.py 2>&1 | grep -E 'OK|PASSED|Error|assert'

## 2. Dataset audit (first call reads the 33 GB CSV once and caches the needed rows, ~20–40 min)

In [ ]:
!python dataset_audit.py --csv "$CSV" --out "$WORK/audit" --cache "$WORK/cache" --max_per_patient $MAXPP
from IPython.display import Markdown; Markdown(open(f'{WORK}/audit/audit.md').read())

## 3. Experiments (resumable), in order of importance
Every run writes one JSON; re-running a cell skips finished runs. Progress: `work/results/logs/run_log.txt`.
* **Tier 1 – essential** (main table + privacy): CT-HTTPS full model and all baselines.
* **Tier 2 – needed for the ablation tables**: other component variants, window length, transfer.
* **Tier 3 – reviewer-proofing**: sensitivity analyses and privacy–utility trade-off.

In [ ]:
!python ablation.py --csv "$CSV" --out "$WORK/results" --cache "$WORK/cache" --seeds $SEEDS --max_per_patient $MAXPP --families A E --variants full mean linear mlp gru lstm transformer   # Tier 1 – essential

In [ ]:
!python ablation.py --csv "$CSV" --out "$WORK/results" --cache "$WORK/cache" --seeds $SEEDS --max_per_patient $MAXPP --families A B C   # Tier 2 – ablation tables

In [ ]:
!python ablation.py --csv "$CSV" --out "$WORK/results" --cache "$WORK/cache" --seeds $SEEDS --max_per_patient $MAXPP --families S P     # Tier 3 – sensitivity + privacy-utility

## 4. Report: every table, figure and quoted number, built from the logs

In [ ]:
!python make_report.py --results "$WORK/results" --audit "$WORK/audit" --out "$WORK/report" --zip "$WORK/results_summary.zip"
from IPython.display import Markdown; Markdown(open(f'{WORK}/report/REPORT.md').read())

**Done.** Send `MyDrive/ct_https/work/results_summary.zip` (and only that file).